In [0]:
import torch
import torch.optim as optim

# Import our custom models
from models.wfdb_resnet import WFDBResNetLSTM
from models.loss import FocalLoss
from data.wfdb_dataset import WFDBDataset
import os
from tqdm import tqdm
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import mlflow
import mlflow.pytorch
from mlflow.models.signature import infer_signature


In [0]:
# ONE-TIME: Convert 45K individual WFDB files into a single .pt file
# This trades 45K random cloud storage reads for 1 sequential read (~10-20s)
# Run this cell ONCE — the .pt file persists on Volumes across cluster restarts.

import time
from pathlib import Path
import wfdb
import pandas as pd

PT_PATH = "/Volumes/source_sys/raw_sickbay/converted/longke/wfdb/dataset_cache.pt"
RAW_DIR = "/Volumes/source_sys/raw_sickbay/converted/longke/wfdb"

if os.path.exists(PT_PATH):
    print(f"Cached dataset already exists at {PT_PATH} — skipping conversion.")
else:
    print("Converting WFDB files to single .pt cache (one-time operation)...")
    start = time.time()

    # Load SNOMED code mapping
    csv_path = os.path.join(RAW_DIR, "ConditionNames_SNOMED-CT.csv")
    df = pd.read_csv(csv_path)
    unique_codes = df["Snomed_CT"].astype(str).unique().tolist()
    code_to_idx = {code: idx for idx, code in enumerate(unique_codes)}
    num_classes = len(unique_codes)
    print(f"  {num_classes} SNOMED-CT classes")

    # Find all records
    mat_files = sorted(Path(RAW_DIR).rglob("*.mat"))
    print(f"  {len(mat_files)} .mat files found")

    signals_list = []
    labels_list = []
    skipped = 0

    for i, mat_path in enumerate(mat_files):
        if i % 5000 == 0:
            print(f"  Processing {i}/{len(mat_files)}...")
        try:
            record_path = str(mat_path)[:-4]  # strip .mat
            record = wfdb.rdrecord(record_path)
            sig = np.nan_to_num(record.p_signal).T  # (12, seq_len)

            # Build multi-hot label
            label = np.zeros(num_classes, dtype=np.float32)
            for comment in record.comments:
                if comment.startswith("Dx:"):
                    for code in comment.replace("Dx:", "").strip().split(","):
                        code = code.strip()
                        if code in code_to_idx:
                            label[code_to_idx[code]] = 1.0
                    break

            signals_list.append(torch.FloatTensor(sig))
            labels_list.append(torch.FloatTensor(label))
        except Exception as e:
            print(f"Error loading {mat_path}: {e}")
            skipped += 1

    elapsed = time.time() - start
    print(f"  Done! {len(signals_list)} records loaded, {skipped} skipped, in {elapsed:.1f}s")

    # Save as single .pt file
    print(f"  Saving to {PT_PATH}...")
    torch.save({
        "signals": signals_list,
        "labels": labels_list,
        "unique_codes": unique_codes,
        "code_to_idx": code_to_idx,
        "num_classes": num_classes,
    }, PT_PATH)
    size_gb = os.path.getsize(PT_PATH) / 1e9
    print(f"  Saved! File size: {size_gb:.2f} GB")

In [0]:
import random
from torch.utils.data import Dataset

class CachedWFDBDataset(Dataset):
    """
    Loads the pre-converted .pt cache into RAM.
    All I/O happens once at init (~10-20s for 5 GB sequential read).
    __getitem__ is pure memory access — zero disk I/O during training.
    """

    def __init__(self, pt_path, is_train=False):
        print(f"Loading cached dataset from {pt_path}...")
        start = time.time()
        cache = torch.load(pt_path, weights_only=False)
        elapsed = time.time() - start

        self.signals = cache["signals"]   # list of (12, seq_len) tensors
        self.labels = cache["labels"]     # list of (num_classes,) tensors
        self.num_classes = cache["num_classes"]
        self.unique_snomed_codes = cache["unique_codes"]
        self.is_train = is_train

        print(f"  Loaded {len(self.signals)} records in {elapsed:.1f}s")
        print(f"  {self.num_classes} classes, is_train={is_train}")

    def __len__(self):
        return len(self.signals)

    def __getitem__(self, idx):
        signals = self.signals[idx].clone()  # (12, seq_len)
        label = self.labels[idx]

        # Data augmentation during training
        if self.is_train:
            # Random Gaussian noise (simulates baseline wander / muscle artifacts)
            if random.random() < 0.5:
                noise = torch.randn_like(signals) * 0.05
                signals = signals + noise
            # Random lead masking (zero out one lead)
            if random.random() < 0.3:
                lead = random.randint(0, 11)
                signals[lead, :] = 0.0

        return signals, label

In [0]:


def pad_collate(batch):
    """
    Custom collate_fn to pad variable-length 1D ECG tensors.
    """
    tensors = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    # Transpose for pad_sequence: (channels, seq_len) -> (seq_len, channels)
    tensors_transposed = [t.transpose(0, 1) for t in tensors]

    # Pad them! (Finds the longest in the batch and pads the rest with 0)
    padded_tensors = pad_sequence(
        tensors_transposed, batch_first=True, padding_value=0.0
    )

    # Transpose back: (batch, max_seq_len, channels) -> (batch, channels, max_seq_len)
    padded_tensors = padded_tensors.transpose(1, 2)

    # Stack labels
    labels = torch.stack(labels)

    return padded_tensors, labels



In [0]:
# --- Data Loading ---
# Load cached dataset into RAM and create DataLoaders.
# This only needs to run once per session — re-run training cells without repeating this.

full_dataset_train = CachedWFDBDataset(PT_PATH, is_train=True)
full_dataset_val = CachedWFDBDataset(PT_PATH, is_train=False)

train_size = int(0.8 * len(full_dataset_train))
val_size = len(full_dataset_train) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, _ = torch.utils.data.random_split(
    full_dataset_train, [train_size, val_size], generator=generator
)
_, val_dataset = torch.utils.data.random_split(
    full_dataset_val, [train_size, val_size], generator=generator
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    collate_fn=pad_collate,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=4,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    collate_fn=pad_collate,
    persistent_workers=True,
    pin_memory=True,
    prefetch_factor=4,
)

num_classes = full_dataset_train.num_classes
print(f"\nTrain: {len(train_dataset)} samples, Val: {len(val_dataset)} samples")
print(f"Batches per epoch: {len(train_loader)} train, {len(val_loader)} val")

In [0]:
# --- Model & Optimizer Setup ---

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

model = WFDBResNetLSTM(num_classes=num_classes)
model = model.to(device)
model = torch.compile(model)

criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

NUM_EPOCHS = 5
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
scaler = torch.amp.GradScaler("cuda")

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [0]:
# --- Training Loop ---
import tempfile

tmp_dir = tempfile.mkdtemp()
best_model_path = os.path.join(tmp_dir, "best_wfdb_resnet_lstm.pth")
best_val_loss = float("inf")

mlflow.set_experiment("/Shared/ekg_classifier")
with mlflow.start_run() as run:
    mlflow.log_params({
        "learning_rate": 0.0001,
        "num_epochs": NUM_EPOCHS,
        "model_type": "wfdb_resnet_lstm",
        "optimizer": "Adam",
        "scheduler": "CosineAnnealingLR",
        "batch_size": 128,
        "mixed_precision": True,
    })

    patience = 3
    epochs_no_improve = 0

    print(f"\nStarting training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Train]")
        for inputs, targets in train_pbar:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            predicted = (outputs.data > 0.0).float()
            correct += (predicted == targets).sum().item()

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            train_pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        avg_loss = running_loss / len(train_loader)
        total_labels = len(train_loader.dataset) * num_classes
        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Loss: {avg_loss:.4f}, Train Acc: {100 * correct / total_labels:.2f}%")

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for val_inputs, val_targets in tqdm(val_loader, desc=f"Epoch {epoch + 1}/{NUM_EPOCHS} [Val]"):
                val_inputs = val_inputs.to(device, non_blocking=True)
                val_targets = val_targets.to(device, non_blocking=True)

                with torch.amp.autocast("cuda"):
                    outputs = model(val_inputs)
                    loss = criterion(outputs, val_targets)

                predicted = (outputs.data > 0.0).float()
                val_correct += (predicted == val_targets).sum().item()
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = 100 * val_correct / (len(val_loader.dataset) * num_classes)
        print(f"  Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.2f}%")

        mlflow.log_metrics({"train_loss": avg_loss, "val_loss": avg_val_loss, "val_accuracy": avg_val_acc}, step=epoch)
        scheduler.step()

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  --> Checkpoint saved (val_loss improved)")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping after {epoch + 1} epochs.")
                break

    print("Training complete.")
    print(f"MLflow run ID: {run.info.run_id}")

In [0]:
# --- Threshold Calibration & Model Registration ---
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

# Reload best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
print("Loaded best checkpoint for calibration.")

# Collect predictions on validation set
all_probs, all_targets = [], []
with torch.no_grad():
    for val_inputs, val_targets in tqdm(val_loader, desc="Calibrating"):
        val_inputs = val_inputs.to(device, non_blocking=True)
        with torch.amp.autocast("cuda"):
            outputs = model(val_inputs)
        all_probs.append(torch.sigmoid(outputs).cpu().numpy())
        all_targets.append(val_targets.numpy())

all_probs = np.vstack(all_probs)
all_targets = np.vstack(all_targets)

# Find optimal threshold
best_threshold, best_f1 = 0.5, 0.0
for thresh in np.arange(0.10, 0.95, 0.05):
    score = f1_score(all_targets, (all_probs > thresh).astype(float), average="macro", zero_division=0)
    if score > best_f1:
        best_f1, best_threshold = score, thresh

print(f"Optimal threshold: {best_threshold:.2f} (Macro F1: {best_f1:.4f})")

# Compute AUC metrics
try:
    roc_auc = roc_auc_score(all_targets, all_probs, average="macro")
    pr_auc = average_precision_score(all_targets, all_probs, average="macro")
    print(f"Macro ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}")
except Exception as e:
    roc_auc, pr_auc = None, None
    print(f"Could not calculate AUC: {e}")

# Log calibration results and register model in the same MLflow run
with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_metric("optimal_threshold", best_threshold)
    mlflow.log_metric("best_macro_f1", best_f1)
    if roc_auc and not np.isnan(roc_auc):
        mlflow.log_metric("final_roc_auc", float(roc_auc))
    if pr_auc and not np.isnan(pr_auc):
        mlflow.log_metric("final_pr_auc", float(pr_auc))

    # Log threshold as artifact
    thresh_path = os.path.join(tmp_dir, "optimal_threshold.txt")
    with open(thresh_path, "w") as f:
        f.write(str(best_threshold))
    mlflow.log_artifact(thresh_path)
    mlflow.log_artifact(best_model_path)

    # Log and register model
    print("\nRegistering model to Unity Catalog...")
    sample_input = next(iter(val_loader))[0][0:2].numpy()
    model.to("cpu")
    sample_output = model(torch.FloatTensor(sample_input)).detach().numpy()
    signature = infer_signature(sample_input, sample_output)

    mlflow.pytorch.log_model(
        pytorch_model=model,
        name="ekg_crnn_model",
        signature=signature,
        input_example=sample_input,
        pip_requirements="requirements.txt",
        registered_model_name="biomedicalinformatics_analytics.prepared.EKG_CRNN_Classifier",
    )
    print("Model registered successfully!")